In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append('..')
from dataset import MDPTeleopDataset, SeqTeleopDataset

%load_ext autoreload
%autoreload 2

In [ ]:
dataset = SeqTeleopDataset('../data/lqr_optimal/task_[0.001 0.001]_policy_[0.002 0.005]')
len(dataset)

## Position Time Plot

In [ ]:
index = 0

puppet, goal = np.split(dataset.traj_states[index], 2, axis=-1)
error = puppet - goal
error = error[:-1]

scaler = 0.001
master = dataset.traj_actions[index].copy() #- np.sign(error) * np.array([-5e-2, 0, 0])
master *= scaler

In [ ]:
robot_pos = puppet[:-1]
human_pos = np.cumsum(master, axis=0) + robot_pos[0]

In [ ]:
t = np.arange(robot_pos.shape[0])
labels = ["x", "y"]

fig, axs = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

for i, ax in enumerate(axs):
    ax.plot(t, human_pos[:, i], label="master", color="C0")
    ax.plot(t, robot_pos[:, i], label="puppet", color="C1")
    ax.set_ylabel(labels[i])
    ax.grid(True, alpha=0.3)

axs[0].legend(loc="best")
axs[-1].set_xlabel("sample")
plt.tight_layout()
plt.show()

## Position Time Plot

In [ ]:
from model.utils import resample_sequence
labels = ["x", "y"]

# compute global target length across all trajectories
lengths = []
for idx in range(len(dataset.traj_states)):
    puppet, goal = np.split(dataset.traj_states[idx], 2, axis=-1)
    master = dataset.traj_actions[idx].copy()
    robot_pos = puppet[:-1]
    human_pos = np.cumsum(master, axis=0) + robot_pos[0]
    lengths.append(max(robot_pos.shape[0], human_pos.shape[0]))

target_len = max(lengths) if len(lengths) > 0 else 0

fig, axs = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

for index in range(len(dataset.traj_states)):
    puppet, goal = np.split(dataset.traj_states[index], 2, axis=-1)
    error = puppet - goal
    error = error[:-1]

    master = dataset.traj_actions[index].copy()  # - np.sign(error) * np.array([-5e-2, 0, 0])

    robot_pos = puppet[:-1]
    human_pos = np.cumsum(master, axis=0) + robot_pos[0]

    # resample both sequences to the common target length
    robot_res = resample_sequence(robot_pos, target_len)
    human_res = resample_sequence(human_pos, target_len)

    for i, ax in enumerate(axs):
        t = np.arange(target_len)
        ax.plot(t, human_res[:, i], label=f"master_{index}")
        ax.set_ylabel(labels[i])
        ax.grid(True, alpha=0.3)

    axs[0].legend(loc="best")
    axs[-1].set_xlabel("sample")

plt.tight_layout()
plt.show()

## Diff Time Plot

In [ ]:
index = 0

current, goal = np.split(dataset.traj_states[index], 2, axis=-1)
error = current - goal
error = error[:-1]

scaler = 0.001
puppet = np.diff(dataset.traj_states[index][:, :3], axis=0)
master = dataset.traj_actions[index] - np.sign(error) * np.array([-5e-2, 0, 0])
master *= scaler

In [ ]:
t = np.arange(master.shape[0])

fig, axs = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
axes_labels = ['x', 'y']
for i, ax in enumerate(axs):
    ax.plot(t, master[:, i], label='master', color='C0')
    ax.plot(t, puppet[:, i], label='puppet', color='C1')
    ax.set_ylabel(axes_labels[i])
    ax.grid(True)

axs[-1].set_xlabel('sample')
axs[0].legend(loc='best')
plt.tight_layout()
plt.show()

## 2D Traj Plot

In [ ]:
num_trajs = min(len(dataset.traj_states), 20)

# choose grid size
cols = 5
rows = np.ceil(num_trajs / cols).astype(int)

fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i >= num_trajs:
        ax.axis("off")
        continue

    positions = dataset.traj_states[i][:, :2]
    goals = np.unique(dataset.traj_states[i][:, 3:5], axis=0)

    ax.scatter(
        positions[:, 0],
        positions[:, 1],
        c=np.arange(len(positions)),
        cmap="YlOrRd",
        s=10,
        alpha=0.7
    )
    ax.scatter(goals[:, 0], goals[:, 1], color="gray", s=60, marker="o")

    ax.set_title(f"Trajectory {i}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    
    # OpenCV coordinate convention
    ax.invert_xaxis()
    ax.set_aspect("equal", adjustable="box")

plt.tight_layout()
plt.show()

## Learning Curves

In [ ]:
dataset.traj_lengths

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

time_in_seconds = np.array(dataset.traj_lengths) / 200.0
ax.plot(time_in_seconds, marker='o', linestyle='-', linewidth=2, markersize=6)

ax.set_xlabel('Trajectory Index')
ax.set_ylabel('Duration (seconds)')
ax.set_title('Trajectory Duration')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Effort

In [ ]:
index = 0

positions = dataset.traj_states[index][:, :2]
goals = dataset.traj_states[index][:, 3:5]
distances = np.linalg.norm(positions - goals, axis=1)

actions = dataset.traj_actions[index]

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(distances, marker='o', linestyle='-', linewidth=2, markersize=6)
ax.plot(np.linalg.norm(actions, axis=1), marker='x', linestyle='--', linewidth=2, markersize=6)

ax.set_xlabel('Sample')
ax.set_ylabel('Distance (m)')
ax.set_title(f'Distance to Goal - Trajectory {index}')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
from model.utils import resample_sequence
import numpy as np
import matplotlib.pyplot as plt

# Plot action norms for all trajectories on the same axes
resample = True  # set False to plot raw lengths
num_trajs = len(dataset.traj_actions)
lengths = [dataset.traj_actions[i].shape[0] for i in range(num_trajs)]
target_len = max(lengths) if (len(lengths) > 0 and resample) else None

fig, ax = plt.subplots(figsize=(10, 6))
for i in range(num_trajs):
    actions = dataset.traj_actions[i].copy()
    norms = np.linalg.norm(actions, axis=1)
    if resample and target_len is not None:
        norms = resample_sequence(norms, target_len)
        x = np.arange(target_len)
    else:
        x = np.arange(norms.shape[0])
    ax.plot(x, norms, label=f'traj_{i}')

ax.set_xlabel('sample')
ax.set_ylabel('action norm')
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize='small')
plt.tight_layout()
plt.show()